# XML → tomograms

**Use this when** you already have an alignment from **Warp / M / MissAlignment** (a tilt-series XML) and want TeamTomo tomograms. The reconstruction runs in *Warp-compatible* mode: given the same XML it reproduces WarpTools `ts_reconstruct` (correlation 0.995 with Warp's own tomogram on EMPIAR-10499 series 00254).

**You need** a Warp project folder (with `warp_tiltseries/<series>.xml`, and either Warp's per-tilt averages in `warp_frameseries/average/` or the tilt stack in `warp_tiltseries/tiltstack/`). Set the config file below; everything else follows from it.

**Output** in `teamtomo_output/<series>/`: the tomogram (`<series>_<pixel>Apx.mrc`) and `run.json` (settings, package versions, timings).

In [ ]:
CONFIG = "configs/xml_to_tomograms_warp_project.yaml"   # everything about the run is in this file (inputs, outputs, options)
SERIES = "00254"          # which tilt series to run here (batch.py runs them all)
OVERRIDES = {}            # optional, e.g. {"reconstruction": {"pixel_spacing": 5.0}}

In [ ]:
import sys, json, numpy as np, mrcfile, matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import gaussian_filter
sys.path.insert(0, ".")
from teamtomo_workflows import load_config, discover, xml_to_tomograms

cfg = load_config(CONFIG, OVERRIDES)
series = discover(cfg)
print(f"{cfg['workflow']}: {len(series)} tilt series found ({series[0].source if series else '-'})")
for s in series:
    print(f"  {s.name:10s} {len(s.tilt_angles)} tilts, {s.tilt_angles.min():.0f}..{s.tilt_angles.max():.0f} deg | stack: {'yes' if s.stack else 'no'} | "
          f"Warp XML: {'yes' if s.xml else 'no'} | full-size averages: {'yes' if s.averages else 'no'}")
s = next(x for x in series if x.name == SERIES)

## Reconstruct
Reads the XML (alignment, per-tilt CTF and dose, Warp's local grids if any), loads the images at the output pixel size the way Warp does, and reconstructs.

In [ ]:
rec = xml_to_tomograms(cfg, s, cfg["compute"]["device"])
print(f"{rec['status']} in {rec['total_s']} s  ->  {Path(cfg['output']['root']) / s.name}")
print("steps (s):", rec["timings_s"])
print("results:", {k: v for k, v in rec["results"].items() if not isinstance(v, list) or len(v) < 4})

In [ ]:
def show_tomogram(path, title=""):
    v = mrcfile.read(path).astype(np.float32); z, y = v.shape[0] // 2, v.shape[1] // 2
    fig, ax = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={"width_ratios": [1, 1.6]})
    for a, sl, t in ((ax[0], v[z - 2:z + 3].mean(0), f"xy slice z={z}"), (ax[1], v[:, y - 2:y + 3].mean(1), f"xz slice y={y}")):
        sl = gaussian_filter(sl, 1); lo, hi = np.percentile(sl, (1, 99))
        a.imshow(sl, cmap="gray", vmin=lo, vmax=hi, origin="lower", aspect="equal" if a is ax[0] else "auto"); a.set_title(f"{title} {t}"); a.set_xticks([]); a.set_yticks([])
    plt.tight_layout()
run_dir = Path(cfg["output"]["root"]) / s.name
show_tomogram(run_dir / rec["outputs"]["tomogram"], s.name)

## All tilt series at once
The same code from the command line, one series after another on one GPU (resumes where it stopped; a failed series is logged and skipped):
```bash
python batch.py --config configs/xml_to_tomograms_warp_project.yaml
```